In [0]:
from pyspark.sql import functions as f,Window
from pyspark.sql.types import *
import re
from datetime import datetime

spr = spark.read.table('madstagescd2.madstage.goldcustomersorders')

spr = spr.groupBy('customer_name', 'city', 'customer_segment').agg(
    f.count(f.col('order_id')).alias('NumberOfOrders'),
    f.sum(f.col('quantity')).alias('TotalQuantityPurchased'),
    f.sum(f.col('order_amount')).alias('TotalOrderAmount'),
    f.avg(f.col('order_amount')).alias('AverageOrderAmount')  
)

spr = spr.withColumn('TopCustomersBySales',f.max(f.col('TotalOrderAmount')).over(Window.partitionBy()))

spr.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('CustomerSalesReport')

print('CustomerSalesReport Generated')